# Notebook 3 — Baseline MIL, Model Definitions, Pooling and Weakly Supervised Losses

## Purpose of this notebook

This notebook defines the main model and loss components used for weakly supervised Video Anomaly Detection.

Notebook 1 prepares metadata and dataset splits. Notebook 2 converts those splits into model-ready feature manifests and dataloaders. Notebook 3 then defines the baseline MIL model, model-checking utilities, weakly supervised ranking losses, pooling functions, and evaluation helpers used by the later training and evaluation notebooks.

The main outputs of this notebook are:

- the baseline MIL segment scoring model;
- the attention-based MIL model definition used for the improved method;
- the integrated CLIP semantic-head model definition used for the extra semantic extension;
- weakly supervised ranking loss functions;
- top-k and max pooling helpers;
- validation metric helpers for AUC and AP;
- a model registry describing which models are trained and evaluated later.

## Pipeline position

```text
Notebook 1: metadata, splits and configuration
        ↓
Notebook 2: feature bags, manifests and dataloaders
        ↓
Notebook 3: model definitions, pooling functions and losses
        ↓
Notebook 4: model training and checkpoint saving
        ↓
Notebook 5: final evaluation, visualisation and report figures
```

## Weakly supervised MIL formulation

The project uses video-level labels only. Each video is represented as a bag of temporal segment features:

```text
video bag = [segment_1, segment_2, ..., segment_32]
```

Each segment receives an anomaly score. The model is trained so that anomalous videos receive higher pooled anomaly scores than normal videos.

In this formulation:

```text
normal video → all segments should have low anomaly scores
anomalous video → at least some segments should have high anomaly scores
```

Because exact frame-level anomaly boundaries are not used for training, Multiple Instance Learning is a suitable framework for learning from these weak labels.

## Main configuration used by this notebook

The submitted run uses:

```text
feature mode = 3d
input feature dimension = 512
number of temporal segments = 32
main pooling method = top-k pooling
top-k value = 4
baseline model = feed-forward MIL segment scorer
improved model = temporal attention MIL
extra extension = integrated CLIP semantic head
```

This notebook defines the components required for training and evaluation. The actual final model training is performed in Notebook 4.


# Notebook configuration handoff

This notebook starts by loading the shared session file created earlier in the pipeline:

```text
artifacts/task1_session.json
```

The session file stores the project paths, dataset mode, feature mode, feature dimensions, number of temporal segments, and CLIP-related configuration. Loading this file ensures that Notebook 3 uses the same settings as Notebooks 1 and 2.

The bootstrap cell below restores:

- project root paths;
- metadata and split directories;
- feature directories;
- dataset mode: UCF, XD, or both;
- feature mode: 2D, 3D, or fusion;
- number of temporal segments;
- MIL input dimension;
- CLIP semantic configuration;
- output and checkpoint directories.

This keeps the multi-notebook pipeline consistent and avoids manually redefining paths or feature settings.


In [1]:
# Bootstrap from Task 1 session — run first on a fresh kernel
import json
import os
import platform
from pathlib import Path

import torch

def _submission_root() -> Path:
    env = os.environ.get("VAD_SUBMISSION_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

SUBMISSION_ROOT = _submission_root()
SESSION_PATH = SUBMISSION_ROOT / "artifacts" / "task1_session.json"
if not SESSION_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SESSION_PATH}. Run Task 1 to completion and execute its session-export cell, "
        "or set VAD_SUBMISSION_ROOT to your dev_submission folder."
    )

sess = json.loads(SESSION_PATH.read_text(encoding="utf-8"))

PROJECT_ROOT = Path(sess["project_root"])
PROCESSED_ROOT = sess["processed_root"]
DATA_ROOT = sess["data_root"]
XD_VIOLENCE_ROOT = sess["xd_violence_root"]
I3D_FEATURES_DIR = Path(sess["i3d_features_dir"])
METADATA_DIR = Path(sess["metadata_dir"])
SPLITS_DIR = Path(sess["splits_dir"])
FEATURES_DIR = Path(sess["features_dir"])
FEATURES_DIR_XD = Path(sess["features_dir_xd"])
FEATURES_3D_DIR = Path(sess["features_3d_dir"])
FEATURES_FUSION_DIR = Path(sess["features_fusion_dir"])
RESULTS_DIR = Path(sess["results_dir"])

USE_RUNS_DIR = bool(sess["use_runs_dir"])
RUN_TAG = sess["run_tag"]
CFG = sess["cfg"]
SEED = int(sess["seed"])
NUM_SEGMENTS = int(sess["num_segments"])
DATASET_MODE = sess["dataset_mode"]
USE_FUSION = bool(sess["use_fusion"])
USE_3D_FEATURES = bool(sess["use_3d_features"])

# Extra-credit switch from Notebook 1.
USE_CLIP_TEXT_EXTRA_CREDIT = bool(sess.get("use_clip_text_extra_credit", True))
ANOMALY_LABEL_SET = sess.get("anomaly_label_set", [
    "normal",
    "abuse",
    "arrest",
    "arson",
    "assault",
    "burglary",
    "explosion",
    "fighting",
    "road accident",
    "robbery",
    "shooting",
    "shoplifting",
    "stealing",
    "vandalism",
    "violence",
])

FEATURE_DIM = int(sess["feature_dim"])
FEATURE_DIM_3D = int(sess["feature_dim_3d"])
XD_FEATURE_DIM = int(sess["xd_feature_dim"])
XD_SEGMENTS_RAW = int(sess["xd_segments_raw"])
XD_STREAM_OUT_DIM = int(sess["xd_stream_out_dim"])
FRAMES_PER_SEGMENT = int(sess["frames_per_segment"])
RESIZE_HW = tuple(sess["resize_hw"])
CLIP_LEN = int(sess["clip_len"])
CLIP_STRIDE = int(sess["clip_stride"])
TRAIN_RATIO = float(sess["train_ratio"])
VAL_RATIO = float(sess["val_ratio"])
TEST_RATIO = float(sess["test_ratio"])
IN_COLAB = bool(sess["in_colab"])
PHASE_ACTIVE = int(sess.get("phase_active", 2))

_ = platform.system()
print("Loaded session:", SESSION_PATH)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)

FORCE_CPU = False

def select_torch_device(force_cpu: bool = False) -> torch.device:
    if force_cpu:
        print("TORCH_DEVICE: cpu (FORCE_CPU=True)")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        print("TORCH_DEVICE: cpu (CUDA not available)")
        return torch.device("cpu")
    try:
        x = torch.randn(1, 3, 224, 224, device="cuda")
        w = torch.randn(64, 3, 7, 7, device="cuda")
        _ = torch.nn.functional.conv2d(x, w, padding=3)
        del _, w, x
        torch.cuda.empty_cache()
        print("TORCH_DEVICE: cuda (smoke test passed)")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA reported but ops failed ({str(e)[:120]}...); using CPU")
        return torch.device("cpu")

TORCH_DEVICE = select_torch_device(FORCE_CPU)
device = TORCH_DEVICE

if USE_FUSION:
    MIL_INPUT_DIM = FEATURE_DIM + FEATURE_DIM_3D
elif USE_3D_FEATURES:
    MIL_INPUT_DIM = FEATURE_DIM_3D
else:
    MIL_INPUT_DIM = FEATURE_DIM

MIL_DUMMY_T = 32 if (USE_3D_FEATURES or USE_FUSION) else NUM_SEGMENTS

def ucf_feature_dir() -> Path:
    if USE_FUSION:
        return FEATURES_FUSION_DIR
    return FEATURES_3D_DIR if USE_3D_FEATURES else FEATURES_DIR

print("Bootstrap OK — continue with Section 6 onward.")

print("USE_CLIP_TEXT_EXTRA_CREDIT:", USE_CLIP_TEXT_EXTRA_CREDIT)
print("ANOMALY_LABEL_SET size:", len(ANOMALY_LABEL_SET))
print("MIL_INPUT_DIM:", MIL_INPUT_DIM)


Loaded session: /scratch/VAD/artifacts/task1_session.json
PROJECT_ROOT: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217
PROCESSED_ROOT: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed
TORCH_DEVICE: cuda (smoke test passed)
Bootstrap OK — continue with Section 6 onward.
USE_CLIP_TEXT_EXTRA_CREDIT: True
ANOMALY_LABEL_SET size: 15
MIL_INPUT_DIM: 512


# 1. Load metadata splits

This section loads the train, validation and test split files created earlier in the pipeline.

For UCF-Crime, the required files are:

```text
train_df.pkl
val_df.pkl
test_df.pkl
```

For XD-Violence, when `DATASET_MODE` is set to `xd` or `both`, the required files are:

```text
xd_train_df.pkl
xd_val_df.pkl
xd_test_df.pkl
```

These files contain clip-level metadata and binary labels. They are used here to rebuild or load the train, validation and test manifests required by the dataloaders.

This step ensures that the model and loss definitions are checked using the same train/validation/test data setup used by the later training notebook.


In [2]:
# Load UCF train/val/test splits produced by Task 1 (required on a fresh kernel)
import pandas as pd

for _name in ("train_df", "val_df", "test_df"):
    _p = SPLITS_DIR / f"{_name}.pkl"
    if not _p.exists():
        raise FileNotFoundError(
            f"Missing {_p}. Run Task 1 through the split + session-export cells first."
        )

train_df = pd.read_pickle(SPLITS_DIR / "train_df.pkl")
val_df = pd.read_pickle(SPLITS_DIR / "val_df.pkl")
test_df = pd.read_pickle(SPLITS_DIR / "test_df.pkl")
print("Loaded Task 1 splits:", len(train_df), "train |", len(val_df), "val |", len(test_df), "test")

if DATASET_MODE in ("xd", "both"):
    for _name in ("xd_train_df", "xd_val_df", "xd_test_df"):
        _p = SPLITS_DIR / f"{_name}.pkl"
        if not _p.exists():
            raise FileNotFoundError(
                f"Missing {_p}. In Task 1, run the XD metadata/split cells, then Section 5 "
                "(writes xd_*.pkl next to UCF splits), then session export."
            )
    xd_train_df = pd.read_pickle(SPLITS_DIR / "xd_train_df.pkl")
    xd_val_df = pd.read_pickle(SPLITS_DIR / "xd_val_df.pkl")
    xd_test_df = pd.read_pickle(SPLITS_DIR / "xd_test_df.pkl")
    print(
        "Loaded XD splits:",
        len(xd_train_df),
        "train |",
        len(xd_val_df),
        "val |",
        len(xd_test_df),
        "test",
    )


Loaded Task 1 splits: 1420 train | 190 val | 290 test
Loaded XD splits: 15816 train | 3954 val | 4000 test


# 2. Manifest preparation

This section defines helper functions for building feature manifests.

A manifest connects each clip to its saved `.npy` feature bag and binary label. Each row contains:

```text
clip_id
label
feature_path
num_segments
```

The manifest files are:

```text
train_manifest.csv
val_manifest.csv
test_manifest.csv
```

The model notebooks do not read raw frames directly. They read temporal feature bags from the paths listed in these manifest files.

When both UCF-Crime and XD-Violence are enabled, this section builds combined manifests according to `DATASET_MODE`.


In [3]:
# Define build_feature_manifest() and save_manifests(); columns: clip_id, label, feature_path, num_segments.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
import pandas as pd
from pathlib import Path

import numpy as np

def build_feature_manifest(split_df: pd.DataFrame, features_dir: Path) -> pd.DataFrame:
    rows = []
    for _, row in split_df.iterrows():
        clip_id = row["clip_id"]
        label = int(row["label"])
        fname = clip_id if str(clip_id).endswith(".npy") else f"{clip_id}.npy"
        feature_path = features_dir / fname
        if not feature_path.exists():
            continue
        bag = np.load(feature_path)
        rows.append({"clip_id": clip_id, "label": label, "feature_path": str(feature_path), "num_segments": bag.shape[0]})
    return pd.DataFrame(rows)

def save_manifests(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, features_dir: Path, out_dir: Path) -> None:
    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        manifest = build_feature_manifest(df, features_dir)
        path = out_dir / f"{name}_manifest.csv"
        manifest.to_csv(path, index=False)
        print(f"{name}_manifest.csv:", len(manifest), "rows")

# Run after feature extraction:
# save_manifests(train_df, val_df, test_df, FEATURES_DIR, METADATA_DIR)
print("Run the cell below to write manifest CSVs to METADATA_DIR.")

Run the cell below to write manifest CSVs to METADATA_DIR.


In [4]:
# Write train/val/test_manifest.csv to METADATA_DIR. Run after Section 5 (splits) and feature extraction.
_ucf_dir = ucf_feature_dir()
save_manifests(train_df, val_df, test_df, _ucf_dir, METADATA_DIR)

train_manifest.csv: 1420 rows
val_manifest.csv: 190 rows
test_manifest.csv: 290 rows


In [5]:
# Build train_manifest_df and val_manifest_df from UCF and/or XD according to DATASET_MODE. Required before DataLoader/training.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
_ucf_feat_dir = ucf_feature_dir()
if DATASET_MODE == "ucf":
    train_manifest_df = build_feature_manifest(train_df, _ucf_feat_dir)
    val_manifest_df = build_feature_manifest(val_df, _ucf_feat_dir)
    print("Manifests from UCF. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
elif DATASET_MODE == "xd":
    if "xd_train_df" not in globals() or "xd_val_df" not in globals():
        raise RuntimeError("Run Phase 2 (XD metadata) and Section 2b (XD feature bags) first.")
    train_manifest_df = build_feature_manifest(xd_train_df, FEATURES_DIR_XD)
    val_manifest_df = build_feature_manifest(xd_val_df, FEATURES_DIR_XD)
    print("Manifests from XD. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
elif DATASET_MODE == "both":
    ucf_train = build_feature_manifest(train_df, _ucf_feat_dir)
    ucf_val = build_feature_manifest(val_df, _ucf_feat_dir)
    xd_train_m = build_feature_manifest(xd_train_df, FEATURES_DIR_XD)
    xd_val_m = build_feature_manifest(xd_val_df, FEATURES_DIR_XD)
    train_manifest_df = pd.concat([ucf_train, xd_train_m], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    val_manifest_df = pd.concat([ucf_val, xd_val_m], ignore_index=True)
    print("Manifests from UCF+XD. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
else:
    raise ValueError("DATASET_MODE must be 'ucf', 'xd', or 'both'.")
if DATASET_MODE == "ucf":
    test_manifest_df = build_feature_manifest(test_df, _ucf_feat_dir)
elif DATASET_MODE == "xd":
    test_manifest_df = build_feature_manifest(xd_test_df, FEATURES_DIR_XD)
else:
    test_manifest_df = pd.concat([build_feature_manifest(test_df, _ucf_feat_dir), build_feature_manifest(xd_test_df, FEATURES_DIR_XD)], ignore_index=True)
train_manifest_df.to_csv(METADATA_DIR / "train_manifest.csv", index=False)
val_manifest_df.to_csv(METADATA_DIR / "val_manifest.csv", index=False)
test_manifest_df.to_csv(METADATA_DIR / "test_manifest.csv", index=False)
if len(train_manifest_df) < 50 or len(val_manifest_df) < 20:
    print("Warning: very few clips in manifest. Validation AUC will be unreliable. Run more feature extraction (UCF and/or XD Section 2b), then re-run manifest and DataLoader.")


Manifests from UCF+XD. Train: 17236 | Val: 4144


# 3. Feature dimension verification

This section verifies that all feature bags used by the model have a consistent input dimension.

This is important because the combined dataset can contain feature bags from different sources:

```text
UCF-Crime → R(2+1)D feature bags
XD-Violence → RGB + Flow I3D fused feature bags
```

For the final submitted run, both sources are aligned to:

```text
feature_dim = 512
```

The verification cell checks the train and validation manifests before model creation. If mismatched feature dimensions are found, the notebook stops before training-related components are used.

This prevents hidden feature-shape errors from reaching the model.


In [6]:
# ============================================================================
# FEATURE DIMENSION VERIFICATION FOR MODEL NOTEBOOK
# ============================================================================
# Notebook 3 defines models/losses, so before model sanity checks we verify that
# all feature bags have a consistent input dimension.

import numpy as np
import pandas as pd
from collections import Counter

def inspect_manifest_feature_dims_model_nb(manifest_df: pd.DataFrame, split_name: str, max_examples: int = 2):
    dims = []
    examples = {}

    if manifest_df is None or len(manifest_df) == 0:
        print(f"{split_name}: empty manifest")
        return Counter(), {}

    for _, row in manifest_df.iterrows():
        arr = np.load(row.feature_path)
        if arr.ndim != 2:
            raise ValueError(
                f"{split_name}: expected feature bag [T, D], got {arr.shape} "
                f"for {row.feature_path}"
            )

        d = int(arr.shape[1])
        dims.append(d)

        examples.setdefault(d, [])
        if len(examples[d]) < max_examples:
            examples[d].append(str(row.feature_path))

    dim_counts = Counter(dims)

    print("=" * 70)
    print(f"{split_name.upper()} FEATURE DIMENSION CHECK")
    print("=" * 70)
    print("Dimension counts:", dim_counts)

    for d, paths in examples.items():
        print(f"\nExamples for D={d}:")
        for p in paths:
            print("  ", p)

    if len(dim_counts) > 1:
        raise ValueError(
            f"{split_name}: mixed feature dimensions found: {dict(dim_counts)}. "
            "This will break model training. Fix feature alignment before continuing."
        )

    return dim_counts, examples


train_dim_counts, _ = inspect_manifest_feature_dims_model_nb(train_manifest_df, "train")
val_dim_counts, _ = inspect_manifest_feature_dims_model_nb(val_manifest_df, "val")
test_dim_counts, _ = inspect_manifest_feature_dims_model_nb(test_manifest_df, "test")

if len(train_dim_counts) > 0:
    VERIFIED_INPUT_FEATURE_DIM = int(next(iter(train_dim_counts.keys())))
else:
    VERIFIED_INPUT_FEATURE_DIM = None

print("\nVerified input feature dimension:", VERIFIED_INPUT_FEATURE_DIM)
print("MIL_INPUT_DIM from config:", MIL_INPUT_DIM)

if VERIFIED_INPUT_FEATURE_DIM is not None and int(VERIFIED_INPUT_FEATURE_DIM) != int(MIL_INPUT_DIM):
    raise ValueError(
        f"VERIFIED_INPUT_FEATURE_DIM={VERIFIED_INPUT_FEATURE_DIM} but MIL_INPUT_DIM={MIL_INPUT_DIM}. "
        "Fix the feature config before training."
    )

print("Feature dimension check passed.")

TRAIN FEATURE DIMENSION CHECK
Dimension counts: Counter({512: 17236})

Examples for D=512:
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_xd/Spectre.2015__#01-56-13_01-57-08_label_B2-G-B1__1.npy
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_xd/v=U6GNhmLnjWU__#1_label_A__1.npy
VAL FEATURE DIMENSION CHECK
Dimension counts: Counter({512: 4144})

Examples for D=512:
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_3d/Normal_Videos750_x264.npy
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_3d/Explosion018_x264.npy
TEST FEATURE DIMENSION

# 4. Bag normalisation

This section loads or fits feature normalisation statistics for the temporal feature bags.

The normalisation statistics are computed from the training split only:

```text
mean
standard deviation
feature dimension
number of training clips
number of training segments
```

The same training-set statistics are then applied to validation and test feature bags.

Using training-only normalisation avoids data leakage because validation and test data do not influence the scaling parameters.


In [7]:
# ============================================================================
# BAG NORMALIZATION
# ============================================================================
# Fit mean/std on train bags only. Val/test use the same train statistics.
# This safe version checks feature dimensions before concatenation.

import numpy as np
from collections import Counter

_norm_path = METADATA_DIR / "bag_norm_stats.npz"

if len(train_manifest_df) > 0:
    segments = []
    dims = []

    for _, row in train_manifest_df.iterrows():
        bag = np.load(row.feature_path).astype(np.float32)

        if bag.ndim != 2:
            raise ValueError(f"Expected bag [T, D], got {bag.shape} for {row.feature_path}")

        segments.append(bag)
        dims.append(int(bag.shape[1]))

    dim_counts = Counter(dims)
    print("Bag norm feature dimension counts:", dim_counts)

    if len(dim_counts) > 1:
        raise ValueError(
            f"Cannot fit bag normalization with mixed feature dimensions: {dict(dim_counts)}. "
            "Fix UCF/XD feature alignment first."
        )

    all_segments = np.concatenate(segments, axis=0)

    _mean = all_segments.mean(axis=0).astype(np.float32)
    _std = (all_segments.std(axis=0) + 1e-6).astype(np.float32)

    np.savez(
        _norm_path,
        mean=_mean,
        std=_std,
        feature_dim=np.array([all_segments.shape[1]], dtype=np.int32),
        num_train_clips=np.array([len(train_manifest_df)], dtype=np.int32),
        num_train_segments=np.array([all_segments.shape[0]], dtype=np.int32),
    )

    print("Bag norm fitted:")
    print("  train clips:", len(train_manifest_df))
    print("  train segments:", all_segments.shape[0])
    print("  feature dim:", all_segments.shape[1])
    print("  saved to:", _norm_path)

else:
    print("Train manifest empty; skipping bag norm fit.")

Bag norm feature dimension counts: Counter({512: 17236})
Bag norm fitted:
  train clips: 17236
  train segments: 639909
  feature dim: 512
  saved to: /scratch/VAD/artifacts/metadata/bag_norm_stats.npz


# 5. PyTorch MIL feature dataset

This section defines the PyTorch dataset class used to load feature bags from the manifest files.

`MILFeatureDataset` reads each row of the manifest and loads the corresponding `.npy` feature bag from disk.

Each dataset item contains:

```text
features
label
length
```

where:

- `features` is the temporal segment feature bag;
- `label` is the binary video-level label;
- `length` records the number of valid temporal segments.

If normalisation statistics are available, the dataset applies the training-set mean and standard deviation to each feature bag.


In [8]:
# PyTorch Dataset: loads .npy feature bags; optional norm_stats (mean, std) applied (fit on train, same for val/test).
import torch
from torch.utils.data import Dataset
import numpy as np

class MILFeatureDataset(Dataset):
    def __init__(self, manifest_df, norm_stats=None):
        self.df = manifest_df.reset_index(drop=True)
        self.norm_stats = norm_stats  # (mean, std) each shape (D,) or None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        features = np.load(row.feature_path).astype(np.float32)
        if self.norm_stats is not None:
            mean, std = self.norm_stats
            features = (features - mean) / std
        T = features.shape[0]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(row.label), torch.tensor(T, dtype=torch.long)


# 6. DataLoaders and balanced batch sampling

This section creates the training and validation dataloaders used by the MIL models.

The dataloader setup includes:

```text
MILFeatureDataset
BalancedBatchSampler
collate_pad_bags
train_loader
val_loader
```

The balanced sampler creates batches with approximately equal numbers of normal and anomalous videos. This is important for the ranking loss because the model learns by comparing abnormal video bags against normal video bags.

The collate function pads feature bags within a batch and returns tensors in the format expected by the model:

```text
features: [batch_size, temporal_segments, feature_dim]
labels:   [batch_size]
lengths:  [batch_size]
```

The resulting dataloaders are used for model shape checks and later training.


In [9]:
# Create train_loader and val_loader from MILFeatureDataset. Requires non-empty train_manifest_df and val_manifest_df.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
from torch.utils.data import DataLoader, Sampler

class BalancedBatchSampler(Sampler):
    """Yield batch indices so each batch has (approx) equal normal and anomaly samples."""
    def __init__(self, labels, batch_size, seed=42):
        self.labels = np.asarray(labels)
        self.batch_size = batch_size
        self.seed = seed
        self.normal_idx = np.where(self.labels == 0)[0]
        self.anomaly_idx = np.where(self.labels == 1)[0]

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        n_half = self.batch_size // 2
        n_batches = (len(self.labels) + self.batch_size - 1) // self.batch_size
        for _ in range(n_batches):
            n_n = min(n_half, len(self.normal_idx))
            n_a = min(self.batch_size - n_n, len(self.anomaly_idx))
            if n_n == 0: n_n = min(1, len(self.normal_idx))
            if n_a == 0: n_a = min(1, len(self.anomaly_idx))
            idx_n = rng.choice(self.normal_idx, size=n_n, replace=len(self.normal_idx) < n_n)
            idx_a = rng.choice(self.anomaly_idx, size=n_a, replace=len(self.anomaly_idx) < n_a)
            yield list(np.concatenate([idx_n, idx_a]))

    def __len__(self):
        return (len(self.labels) + self.batch_size - 1) // self.batch_size

def collate_pad_bags(batch):
    """Pad variable-length feature bags to max T in batch; return (feats, labels, lengths)."""
    feats = torch.nn.utils.rnn.pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0.0)
    labels = torch.stack([b[1] for b in batch])
    lengths = torch.stack([b[2] for b in batch])
    return feats, labels, lengths

if len(train_manifest_df) == 0 or len(val_manifest_df) == 0:
    raise ValueError(
        "Manifests are empty. Run Section 7 (feature extraction) first: "
        "extract_and_save_clip_features(train_df/val_df, FEATURES_DIR) to create .npy feature bags."
    )

_norm_path = METADATA_DIR / "bag_norm_stats.npz"
norm_stats = None
if _norm_path.exists():
    _d = np.load(_norm_path)
    norm_stats = (_d["mean"], _d["std"])
    print("Using bag normalization (fit on train).")

train_dataset = MILFeatureDataset(train_manifest_df, norm_stats=norm_stats)
val_dataset = MILFeatureDataset(val_manifest_df, norm_stats=norm_stats)

USE_BALANCED_SAMPLER = True  # Each batch has both classes when possible
batch_size = int(CFG["batch_size"]) if "CFG" in globals() else 8
if USE_BALANCED_SAMPLER and len(train_manifest_df["label"].unique()) > 1:
    bal_sampler = BalancedBatchSampler(train_manifest_df["label"].values, batch_size, seed=SEED)
    train_loader = DataLoader(train_dataset, batch_sampler=bal_sampler, collate_fn=collate_pad_bags)
else:
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_pad_bags)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_pad_bags)

if USE_FUSION:
    _xb, _, _ = next(iter(train_loader))
    assert int(_xb.shape[-1]) == int(FEATURE_DIM + FEATURE_DIM_3D), _xb.shape
    print("Fusion batch shape:", tuple(_xb.shape), "feature_dim=", int(_xb.shape[-1]))

if len(test_manifest_df) > 0:
    test_dataset = MILFeatureDataset(test_manifest_df, norm_stats=norm_stats)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad_bags)
else:
    test_loader = None
    print("test_manifest is empty — test_loader not created (test-set eval skipped).")

Using bag normalization (fit on train).


# 7. Dataloader contract check

This section checks the tensor format produced by the dataloaders.

The expected model input is:

```text
features: [B, T, D]
labels:   [B]
lengths:  [B]
```

where:

- `B` is batch size;
- `T` is the number of temporal segments;
- `D` is the feature dimension.

For the submitted configuration, the expected feature tensor has:

```text
D = 512
```

This check confirms that the dataloader output matches the model input contract before defining or testing the MIL models.


In [10]:
# ============================================================================
# DATALOADER CONTRACT CHECK
# ============================================================================
# Confirms the tensor contract used by Notebook 4 training:
# features: [B, T, D], labels: [B], lengths: [B]

_xb, _yb, _lb = next(iter(train_loader))

print("=" * 70)
print("DATALOADER CONTRACT CHECK")
print("=" * 70)
print("features:", tuple(_xb.shape), "| dtype:", _xb.dtype)
print("labels:", tuple(_yb.shape), "| labels in batch:", _yb.tolist())
print("lengths:", tuple(_lb.shape), "| min/max:", int(_lb.min()), int(_lb.max()))
print("feature dim from loader:", int(_xb.shape[-1]))
print("MIL_INPUT_DIM config:", int(MIL_INPUT_DIM))

if int(_xb.shape[-1]) != int(MIL_INPUT_DIM):
    raise ValueError(
        f"Loader feature dim ({int(_xb.shape[-1])}) != MIL_INPUT_DIM ({int(MIL_INPUT_DIM)}). "
        "Fix feature dimension alignment or config before training."
    )

print("Dataloader contract OK.")

DATALOADER CONTRACT CHECK
features: (4, 32, 512) | dtype: torch.float32
labels: (4,) | labels in batch: [0, 0, 1, 1]
lengths: (4,) | min/max: 32 32
feature dim from loader: 512
MIL_INPUT_DIM config: 512
Dataloader contract OK.


# 8. Model definitions and comparison setup

The following sections define the model classes and helper functions required for the baseline and improved MIL experiments.

This notebook defines:

- the baseline MIL model;
- the attention MIL model with an integrated CLIP semantic head;
- the model registry;
- weakly supervised MIL ranking losses;
- top-k and max pooling helpers;
- validation metric helpers.

The actual training of the baseline and attention models is performed in the training notebook. The final metrics, pooling comparison and visualisations are produced in the evaluation notebook.


# 8a. Baseline MIL model

This section defines the baseline Multiple Instance Learning model.

The baseline model receives a temporal feature bag:

```text
input: [B, T, D]
```

and produces one anomaly logit for each temporal segment:

```text
output: [B, T]
```

The same feed-forward segment scorer is applied to every segment. The segment logits are later pooled into a video-level anomaly score using max pooling or top-k pooling.

The baseline is intentionally simple. It provides a reference model for measuring whether the attention-based model improves performance.


In [11]:
# ============================================================================
# BASELINE MIL MODEL
# ============================================================================
# Simple segment-level MIL scorer.
# Input:  [B, T, D]
# Output: [B, T] segment anomaly logits

if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")

import torch
import torch.nn as nn

class MILModel(nn.Module):
    def __init__(self, feature_dim=2048, input_dim=None, dropout=0.0):
        super().__init__()

        input_dim = input_dim if input_dim is not None else feature_dim

        self.input_proj = (
            nn.Linear(input_dim, feature_dim)
            if input_dim != feature_dim
            else nn.Identity()
        )

        self.net = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)
        scores = self.net(x).squeeze(-1)
        return scores


print("Defined MILModel baseline.")

Defined MILModel baseline.


# 8b. Baseline model shape check

This section passes one dataloader batch through the baseline MIL model.

The purpose is to confirm that:

```text
input features:  [B, T, D]
segment logits:  [B, T]
```

The model should produce one segment-level anomaly logit per temporal segment. This check verifies the model shape before the loss and pooling helpers are used.


In [12]:
# ============================================================================
# MODEL SHAPE SANITY CHECK
# ============================================================================
# Runs a tiny batch through the baseline model to confirm it returns
# segment-level logits with shape [B, T].

with torch.no_grad():
    _xb, _yb, _lb = next(iter(train_loader))
    _B, _T, _D = _xb.shape

    print("=" * 70)
    print("MODEL SHAPE SANITY CHECK")
    print("=" * 70)
    print("Input batch:", tuple(_xb.shape))

    _baseline = MILModel(input_dim=MIL_INPUT_DIM).to(TORCH_DEVICE)
    _baseline_out = _baseline(_xb.to(TORCH_DEVICE))
    print("MILModel output:", tuple(_baseline_out.shape))

    if tuple(_baseline_out.shape) != (_B, _T):
        raise ValueError(f"MILModel output shape mismatch: expected {(_B, _T)}, got {tuple(_baseline_out.shape)}")


    del _baseline, _baseline_out

    if TORCH_DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("Model shape checks passed.")

MODEL SHAPE SANITY CHECK
Input batch: (4, 32, 512)
MILModel output: (4, 32)
Model shape checks passed.


# 8c. Attention MIL with integrated CLIP semantic head

This section defines the improved MIL model used for the attention and CLIP extension experiments.

The attention model receives the same temporal feature bag as the baseline:

```text
input: [B, T, D]
```

It then projects segment features into a hidden space and applies temporal self-attention. This allows each segment to exchange information with other segments in the same video.

The model outputs:

```text
segment_logits
semantic_logits
video_clip_embedding
semantic_targets
attention_weights
```

The segment logits are used for binary anomaly detection. The semantic head is used for the extra CLIP-based extension, where the model learns an additional mapping between video-level features and CLIP text embeddings for anomaly labels.

The CLIP semantic branch is an auxiliary extension. The main binary anomaly detection task still depends on the MIL anomaly scores.


In [13]:
# ============================================================================
# ATTENTION MIL + CLIP SEMANTIC HEAD MODEL
# ============================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class AttentionMILWithCLIPSemanticHead(nn.Module):
    """
    Attention MIL model with an integrated CLIP semantic projection head.

    Outputs:
        segment_logits: [B, T]
        semantic_logits: [B, C]
        video_clip_embedding: [B, clip_dim]

    The CLIP text embeddings are frozen prototypes generated by CLIP's text encoder.
    The model learns to project video-level MIL representations into the same
    semantic space.
    """

    def __init__(
        self,
        text_embeddings,
        feature_dim=512,
        input_dim=None,
        hidden_dim=256,
        num_heads=4,
        dropout=0.3,
    ):
        super().__init__()

        input_dim = input_dim if input_dim is not None else feature_dim

        text_embeddings = torch.as_tensor(text_embeddings, dtype=torch.float32)
        text_embeddings = F.normalize(text_embeddings, dim=-1)

        self.register_buffer("text_embeddings", text_embeddings)

        clip_dim = int(text_embeddings.shape[1])

        self.input_proj = (
            nn.Linear(input_dim, feature_dim)
            if input_dim != feature_dim
            else nn.Identity()
        )

        self.proj = nn.Linear(feature_dim, hidden_dim)

        self.layer1 = TransformerEncoderLayerWithWeights(
            hidden_dim,
            num_heads,
            hidden_dim * 4,
            dropout,
        )

        self.layer2 = TransformerEncoderLayerWithWeights(
            hidden_dim,
            num_heads,
            hidden_dim * 4,
            dropout,
        )

        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

        self.semantic_projection = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, clip_dim),
        )

        # Learnable CLIP-style temperature scale.
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))

        self.return_attention = False

    def masked_mean_pool(self, x, lengths):
        """
        x:       [B, T, H]
        lengths: [B]
        """
        B, T, H = x.shape

        if lengths is None:
            return x.mean(dim=1)

        mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)
        mask = mask.unsqueeze(-1).float()

        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        return pooled

    def forward(self, x, lengths=None, need_weights=None, return_semantic=True):
        need_weights = need_weights if need_weights is not None else self.return_attention

        x = self.input_proj(x)
        h = self.proj(x)

        h, _ = self.layer1(h, need_weights=False)
        h, attn_weights = self.layer2(h, need_weights=need_weights)

        segment_logits = self.scorer(h).squeeze(-1)

        video_repr = self.masked_mean_pool(h, lengths)
        video_clip_embedding = self.semantic_projection(video_repr)
        video_clip_embedding = F.normalize(video_clip_embedding, dim=-1)

        semantic_logits = self.logit_scale.exp().clamp(max=100.0) * (
            video_clip_embedding @ self.text_embeddings.T
        )

        if need_weights:
            return segment_logits, semantic_logits, video_clip_embedding, attn_weights

        if return_semantic:
            return segment_logits, semantic_logits, video_clip_embedding

        return segment_logits


print("Defined AttentionMILWithCLIPSemanticHead.")

Defined AttentionMILWithCLIPSemanticHead.


# 8d. Model registry export

This section saves a registry of the models used in the project.

The registry records:

```text
model key
display name
checkpoint filename
whether the model is enabled
short description
```

The model registry helps the later notebooks load and evaluate models consistently. It also makes the final model comparison easier to reproduce.


In [14]:
# ============================================================================
# MODEL REGISTRY EXPORT
# ============================================================================
# Notebook 4 will train the models listed here. This file is also useful for
# explaining the model comparison in the report.

import json

MODEL_REGISTRY = [
    {
        "key": "baseline_mil",
        "display_name": "Baseline MIL",
        "checkpoint": "mil_best_baseline.pth",
        "enabled": True,
        "description": "Feed-forward segment-level MIL scorer with top-k video pooling.",
    },
    {
        "key": "attention_mil",
        "display_name": "Attention MIL",
        "checkpoint": "mil_best_attention.pth",
        "enabled": True,
        "description": "Temporal self-attention model over segment-level spatiotemporal features.",
    },
    {
    "model_key": "attention_clip_semantic",
    "display_name": "Attention MIL + CLIP Semantic Head",
    "description": (
        "Attention MIL model with an auxiliary semantic projection head trained "
        "against CLIP text embeddings."
    ),
}
]

model_registry_path = METADATA_DIR / "model_registry.json"
model_registry_path.write_text(
    json.dumps(MODEL_REGISTRY, indent=2, sort_keys=True),
    encoding="utf-8",
)

print("Saved model registry:", model_registry_path)
display(pd.DataFrame(MODEL_REGISTRY))

Saved model registry: /scratch/VAD/artifacts/metadata/model_registry.json


,key,display_name,checkpoint,enabled,description,model_key
0,baseline_mil,Baseline MIL,mil_best_baseline.pth,True,Feed-forward segment-level MIL scorer with top...,NaN
1,attention_mil,Attention MIL,mil_best_attention.pth,True,Temporal self-attention model over segment-lev...,NaN
2,NaN,Attention MIL + CLIP Semantic Head,NaN,NaN,Attention MIL model with an auxiliary semantic...,attention_clip_semantic


# 9. Ranking loss and pooling helpers

This section defines the weakly supervised MIL loss functions and pooling helpers.

The core MIL assumption is:

```text
anomalous video score > normal video score
```

The ranking loss penalises cases where the highest-scoring anomalous bag is not sufficiently higher than the highest-scoring normal bag.

The Sultani-style loss can include three components:

```text
ranking loss
temporal smoothness penalty
sparsity penalty
```

- The ranking term encourages anomalous videos to score higher than normal videos.
- The sparsity term discourages the model from marking every segment as anomalous.
- The smoothness term encourages neighbouring temporal scores to change gradually.

The notebook also defines pooling helpers for converting segment-level logits into video-level scores:

```text
max pooling
top-k mean pooling
masked top-k pooling
```

The final submitted evaluation uses top-k pooling because it is more stable than relying on a single maximum-scoring segment.


In [15]:
# ============================================================================
# RANKING LOSS + POOLING HELPERS
# ============================================================================
# Sultani-style weakly supervised MIL loss:
# ranking + smoothness over anomaly segments + sparsity over anomaly segments.

import torch

def ranking_loss(anomaly_scores, normal_scores, margin=1.0):
    """Anomaly scores should be higher than normal. Works when batch sizes differ."""
    max_anom = anomaly_scores.max(dim=1)[0]
    max_norm = normal_scores.max(dim=1)[0]
    diff = max_anom.unsqueeze(1) - max_norm.unsqueeze(0)
    loss = torch.relu(margin - diff)
    return loss.mean()


def ranking_loss_sultani(
    anomaly_scores,
    normal_scores,
    margin=1.0,
    lambda_smooth=0.0,
    lambda_sparse=8e-5,
):
    """Full Sultani-style loss: ranking + temporal smoothness + sparsity."""
    rank = ranking_loss(anomaly_scores, normal_scores, margin=margin)

    smooth = torch.tensor(0.0, device=anomaly_scores.device)
    if anomaly_scores.shape[1] > 1:
        smooth = ((anomaly_scores[:, 1:] - anomaly_scores[:, :-1]) ** 2).mean()

    sparse = anomaly_scores.mean()

    return rank + lambda_smooth * smooth + lambda_sparse * sparse


def topk_mean(scores, k=None):
    """Video-level score: mean of top-k segment scores. scores: [B, T]."""
    T = scores.shape[1]
    if k is None:
        k = max(1, T // 8)
    k = min(k, T)
    topk_vals = torch.topk(scores, k, dim=1).values
    return topk_vals.mean(dim=1)


def topk_mean_masked(scores, lengths, k=None):
    """Video-level score over valid segments only. scores: [B, T], lengths: [B]."""
    out = []

    for i in range(scores.shape[0]):
        t = int(lengths[i].item())

        if t <= 0:
            out.append(torch.tensor(0.0, device=scores.device, dtype=scores.dtype))
            continue

        segs = scores[i, :t]
        kk = min(k or max(1, t // 8), t)
        out.append(segs.topk(kk).values.mean())

    return torch.stack(out)


def ranking_loss_sultani_masked(
    anomaly_scores,
    normal_scores,
    len_anom,
    len_norm,
    margin=1.0,
    lambda_smooth=0.0,
    lambda_sparse=8e-5,
):
    """Sultani loss with variable-length bags: masks padded positions."""
    T = anomaly_scores.shape[1]

    mask_anom = torch.arange(T, device=anomaly_scores.device).unsqueeze(0) < len_anom.unsqueeze(1)
    mask_norm = torch.arange(T, device=normal_scores.device).unsqueeze(0) < len_norm.unsqueeze(1)

    a = anomaly_scores.clone()
    n = normal_scores.clone()

    a[~mask_anom] = -1e9
    n[~mask_norm] = -1e9

    rank = ranking_loss(a, n, margin=margin)

    smooth = torch.tensor(0.0, device=anomaly_scores.device)
    valid_smooth_count = 0

    for i in range(anomaly_scores.shape[0]):
        L = int(len_anom[i].item())
        if L > 1:
            smooth = smooth + ((anomaly_scores[i, 1:L] - anomaly_scores[i, :L-1]) ** 2).mean()
            valid_smooth_count += 1

    if valid_smooth_count > 0:
        smooth = smooth / valid_smooth_count

    sparse = torch.tensor(0.0, device=anomaly_scores.device)
    valid_sparse_count = 0

    for i in range(anomaly_scores.shape[0]):
        L = int(len_anom[i].item())
        if L > 0:
            sparse = sparse + anomaly_scores[i, :L].mean()
            valid_sparse_count += 1

    if valid_sparse_count > 0:
        sparse = sparse / valid_sparse_count

    return rank + lambda_smooth * smooth + lambda_sparse * sparse


def count_model_parameters(model):
    """Return total and trainable parameter counts."""
    total = int(sum(p.numel() for p in model.parameters()))
    trainable = int(sum(p.numel() for p in model.parameters() if p.requires_grad))
    return total, trainable


print("Defined Sultani-style ranking losses, masked top-k pooling, and parameter counter.")

Defined Sultani-style ranking losses, masked top-k pooling, and parameter counter.


# 10. MIL evaluation helpers

This section defines helper functions for evaluating model outputs at video level.

The model predicts segment-level logits. These are pooled into one video-level anomaly score per video, then evaluated using:

```text
AUC
AP
```

AUC measures how well the model separates normal and anomalous videos across thresholds. AP summarises the precision-recall curve and is useful when anomaly detection data is imbalanced.

These helpers are reused by the training and evaluation notebooks to ensure that validation and final testing use consistent scoring logic.


In [16]:
# --- MIL evaluation helpers (in-notebook; no external .py). Used by pooling comparison + Section 9a. ---
# Run this cell after `topk_mean_masked` is defined (loss cell above).
import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score


def max_pool_logits_masked(logits: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    """Max over valid segment logits per video."""
    B = logits.shape[0]
    out = []
    for i in range(B):
        t = int(lengths[i].item())
        if t <= 0:
            out.append(0.0)
        else:
            out.append(logits[i, :t].max().item())
    return torch.tensor(out, device=logits.device, dtype=logits.dtype)


def collect_video_scores_mil(model, loader, device, pooling: str, topk_mean_masked_fn):
    """pooling: 'topk' matches val; 'max' = max logits then sigmoid."""
    if pooling not in ("topk", "max"):
        raise ValueError(pooling)
    model.eval()
    scores_all, labels_all = [], []
    with torch.no_grad():
        for features, labels, lengths in loader:
            features = features.to(device)
            lengths = lengths.to(device)
            out = model(features)
            logits = out[0] if isinstance(out, tuple) else out
            if pooling == "topk":
                vs = torch.sigmoid(topk_mean_masked_fn(logits, lengths)).cpu().numpy()
            else:
                vs = torch.sigmoid(max_pool_logits_masked(logits, lengths)).cpu().numpy()
            scores_all.extend(vs.tolist())
            labels_all.extend(labels.numpy().tolist())
    return np.asarray(scores_all, dtype=np.float64), np.asarray(labels_all, dtype=np.float64)


def video_auc_ap(labels: np.ndarray, scores: np.ndarray):
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan"), float("nan")
    return (
        float(roc_auc_score(labels, scores)),
        float(average_precision_score(labels, scores)),
    )


def segment_expanded_roc_auc_approx(manifest_df, video_scores: np.ndarray, video_labels: np.ndarray) -> float:
    """Segment-expanded ROC AUC — APPROXIMATE (not true frame-level GT)."""
    frame_labels, frame_scores = [], []
    for i in range(len(manifest_df)):
        row = manifest_df.iloc[i]
        bag = np.load(row.feature_path)
        t = int(bag.shape[0])
        lab = int(video_labels[i]) if i < len(video_labels) else int(row.label)
        vid_score = float(video_scores[i]) if i < len(video_scores) else 0.0
        for _ in range(t):
            frame_labels.append(lab)
            frame_scores.append(vid_score)
    if len(frame_labels) == 0 or len(set(frame_labels)) < 2:
        return float("nan")
    return float(roc_auc_score(frame_labels, frame_scores))


# 11. Methodology summary

The baseline model is a feed-forward MIL segment scorer. It independently scores each segment and then uses pooling to produce a video-level anomaly score.

The main improvement is the attention-based MIL model. Instead of scoring segments only in isolation, the attention model applies temporal self-attention across the segment sequence. This allows the model to use relationships between earlier and later parts of the video.

The final project also includes an integrated CLIP semantic head as an extra extension. This branch adds semantic supervision by aligning video-level embeddings with CLIP text embeddings for anomaly labels. It supports anomaly label interpretation, but the main task remains binary anomaly detection.

The final model comparison is therefore organised as:

```text
Baseline MIL
        ↓
Attention MIL
        ↓
Attention MIL + CLIP semantic extension
```

The loss and pooling functions defined in this notebook are reused consistently in the later training and evaluation notebooks.
